# Logistic regression

Adnan Masood · University of South Florida

A lender wants to estimate whether a loan will default during a stated period. This small, made-up dataset lets you practice probability estimates; its rows are not real applicants, and it does not decide whether anyone should receive a loan.

**Dataset:** The computer creates 5,000 fictional examples. Each has eight numeric inputs and an outcome label; about 10 in every 100 labels are positive. A stratified split puts 70% of examples in the training set and 30% in the test set while keeping roughly the same positive share in both. Seed 7 makes the random sample repeatable.

**Task:** Fit logistic regression after scaling the input measurements. Inspect five estimated probabilities, then use the held-back test examples to measure ranking with ROC-AUC and probability accuracy with Brier score. These scores answer different questions.

**Compare:** Compare the model's Brier score with a simple reference that predicts the training-set default share for every test example. Both scores use the same held-back rows, so the comparison asks whether the model's probabilities improve on one constant guess.

**Runtime:** This example uses scikit-learn for the model and NumPy through scikit-learn. The scaling rule is learned from training examples only, then reused on test examples. The constant-guess reference also uses only the training outcomes.


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
# Generate fictional examples; the positive label is the event being estimated.
X, y = make_classification(n_samples=5000, n_features=8,
    weights=[0.90, 0.10], class_sep=1.1, random_state=7)
# Hold back 30% for evaluation and preserve the outcome mix in both groups.
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=.30, stratify=y, random_state=7)
# Learn input scaling from training rows, then fit the probability model.
model = make_pipeline(StandardScaler(),
    LogisticRegression(C=1.0, max_iter=2000))
model.fit(Xtr, ytr)
# Evaluate only on held-back rows; ranking and probability error differ.
pd_hat = model.predict_proba(Xte)[:, 1]
print("ROC-AUC:", roc_auc_score(yte, pd_hat))
print("Brier:", brier_score_loss(yte, pd_hat))
print("Baseline Brier:", brier_score_loss(yte, ytr.mean() * __import__('numpy').ones(len(yte))))
print("First five PDs:", pd_hat[:5].round(3))


## Try an experiment

Run the initial example and record its results. Then make one of these changes:

### Make the positive outcome rarer

This creates a new sample with fewer positive outcomes. Check its new share before comparing the scores.

Replace:
```python
weights=[0.90, 0.10]
```
With:
```python
weights=[0.97, 0.03]
```

### Use stronger regularization

The smaller C applies a stronger penalty to large weights. Keep the sample and split fixed, then compare ranking and probability error.

Replace:
```python
C=1.0
```
With:
```python
C=0.1
```

### Create data with 12 inputs

The generator makes a new sample with more input columns; this does not add one controlled signal to the old sample.

Replace:
```python
n_features=8
```
With:
```python
n_features=12
```

### Remove input scaling

Compare with the scaled model on the same split. Scaling changes the units used to fit inputs, not the outcomes.

Replace:
```python
StandardScaler(),
```
With:
```python

```



In [ ]:
# Change C from 1.0 to 0.1 and rerun without changing the data split.
# Record both test scores, then compare each Brier score with the constant-guess reference.


## Break it

Make positive outcomes rarer by changing their share from about 10% to about 3%. Rerun the notebook and compare ranking and both Brier scores. Because the mix of outcomes changed, a different score alone does not prove that one model is better.


In [ ]:
# Change weights=[0.90, 0.10] to weights=[0.97, 0.03], then rerun.
# Check the test-set share of positive outcomes before interpreting the Brier scores.


## What to submit

- A table with ROC-AUC, model Brier score, and constant-guess Brier score for both C settings.
- An explanation of what the first five probabilities estimate and why none of them decides whether to approve a loan.
- The rare-outcome experiment, the new positive share, and one reason its scores are not a fair standalone model comparison.

## Optional extension

You could compare a boosting model or test a review cutoff, but those require extra code. A cutoff also needs stated costs for missed defaults, mistaken declines, and staff review; this notebook does not supply those policy assumptions.
